# Modelo híbrido: factores latentes y popularidad

En este notebook se desarrolla una variante híbrida del recomendador que combina la personalización obtenida mediante factores latentes con una señal de popularidad de las películas.

El objetivo es mejorar la calidad del ranking Top-N manteniendo la capacidad de personalización del modelo de factorización matricial.

In [1]:
import pandas as pd
import numpy as np

In [2]:
ratings = pd.read_csv("../data/raw/ml-latest-small/ratings.csv")
movies = pd.read_csv("../data/raw/ml-latest-small/movies.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## 1. Preparación del modelo de factores latentes

Se prepara nuevamente el modelo de factores latentes utilizando la configuración seleccionada durante la evaluación anterior:

- 20 factores latentes.
- Tasa de aprendizaje de 0.01.
- Regularización de 0.05.
- 21 épocas de entrenamiento.

In [3]:
usuarios = sorted(ratings["userId"].unique())
peliculas = sorted(ratings["movieId"].unique())

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(usuarios)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(peliculas)
}

n_usuarios = len(usuarios)
n_peliculas = len(peliculas)

ratings_modelo = ratings.copy()

ratings_modelo["user_idx"] = ratings_modelo["userId"].map(user_to_idx)
ratings_modelo["movie_idx"] = ratings_modelo["movieId"].map(movie_to_idx)

## 2. Entrenamiento del modelo de factores latentes

El modelo se entrena utilizando todas las valoraciones disponibles, ya que en este notebook el objetivo es construir el recomendador híbrido final y no realizar todavía su evaluación.

In [4]:
n_factores = 20
learning_rate = 0.01
regularizacion = 0.05
n_epocas = 21

np.random.seed(42)

P = np.random.normal(
    scale=0.1,
    size=(n_usuarios, n_factores)
)

Q = np.random.normal(
    scale=0.1,
    size=(n_peliculas, n_factores)
)

bias_usuarios = np.zeros(n_usuarios)
bias_peliculas = np.zeros(n_peliculas)

media_global = ratings["rating"].mean()

In [5]:
for epoca in range(n_epocas):

    datos_epoca = ratings_modelo.sample(
        frac=1,
        random_state=42 + epoca
    )

    for fila in datos_epoca.itertuples():

        u = fila.user_idx
        i = fila.movie_idx
        rating_real = fila.rating

        prediccion = (
            media_global
            + bias_usuarios[u]
            + bias_peliculas[i]
            + np.dot(P[u], Q[i])
        )

        error = rating_real - prediccion

        P_u_anterior = P[u].copy()

        bias_usuarios[u] += learning_rate * (
            error - regularizacion * bias_usuarios[u]
        )

        bias_peliculas[i] += learning_rate * (
            error - regularizacion * bias_peliculas[i]
        )

        P[u] += learning_rate * (
            error * Q[i] - regularizacion * P[u]
        )

        Q[i] += learning_rate * (
            error * P_u_anterior - regularizacion * Q[i]
        )

    print(f"Época {epoca + 1}/{n_epocas}")

Época 1/21
Época 2/21
Época 3/21
Época 4/21
Época 5/21
Época 6/21
Época 7/21
Época 8/21
Época 9/21
Época 10/21
Época 11/21
Época 12/21
Época 13/21
Época 14/21
Época 15/21
Época 16/21
Época 17/21
Época 18/21
Época 19/21
Época 20/21
Época 21/21


## 3. Cálculo de la señal de popularidad

Para incorporar la popularidad al modelo híbrido se calcula el número de valoraciones recibidas por cada película.

Esta señal se transforma posteriormente a una escala entre 0 y 1, permitiendo combinarla con la puntuación obtenida mediante factores latentes.

In [6]:
popularidad = (
    ratings
    .groupby("movieId")
    .size()
)

popularidad_normalizada = (
    (popularidad - popularidad.min())
    /
    (popularidad.max() - popularidad.min())
)

popularidad_normalizada.head()

movieId
1    0.652439
2    0.332317
3    0.155488
4    0.018293
5    0.146341
dtype: float64

In [7]:
def obtener_score_popularidad(movie_id):

    return popularidad_normalizada.get(
        movie_id,
        0.0
    )

In [ ]:
def obtener_score_mf(usuario_id, pelicula_id):

    u = user_to_idx[usuario_id]
    i = movie_to_idx[pelicula_id]

    prediccion = (
        media_global
        + bias_usuarios[u]
        + bias_peliculas[i]
        + np.dot(P[u], Q[i])
    )

    prediccion = np.clip(
        prediccion,
        0.5,
        5.0
    )

    score_normalizado = (
        (prediccion - 0.5)
        /
        (5.0 - 0.5)
    )

    return score_normalizado

## 5. Construcción del score híbrido

El modelo híbrido combina la puntuación personalizada obtenida mediante factores latentes con la popularidad de cada película.

El parámetro `alpha` determina el peso otorgado a cada componente. Un valor próximo a 1 concede mayor importancia a la personalización, mientras que un valor próximo a 0 concede mayor importancia a la popularidad.

In [9]:
def obtener_score_hibrido(
    usuario_id,
    pelicula_id,
    alpha=0.5
):

    score_mf = obtener_score_mf(
        usuario_id,
        pelicula_id
    )

    score_pop = obtener_score_popularidad(
        pelicula_id
    )

    score_hibrido = (
        alpha * score_mf
        +
        (1 - alpha) * score_pop
    )

    return score_hibrido

## 6. Generación de recomendaciones híbridas

A partir del score híbrido se generan recomendaciones para cada usuario excluyendo las películas que ya ha valorado.

Las películas candidatas se ordenan de mayor a menor puntuación híbrida y se seleccionan las primeras posiciones del ranking.

In [10]:
def recomendar_hibrido(
    usuario_id,
    n=10,
    alpha=0.5
):

    vistas = set(
        ratings.loc[
            ratings["userId"] == usuario_id,
            "movieId"
        ]
    )

    scores = []

    for pelicula_id in peliculas:

        if pelicula_id in vistas:
            continue

        score = obtener_score_hibrido(
            usuario_id,
            pelicula_id,
            alpha=alpha
        )

        scores.append(
            (pelicula_id, score)
        )

    scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top_n = scores[:n]

    resultado = pd.DataFrame(
        top_n,
        columns=["movieId", "score_hibrido"]
    )

    resultado = resultado.merge(
        movies,
        on="movieId",
        how="left"
    )

    return resultado[
        [
            "movieId",
            "title",
            "genres",
            "score_hibrido"
        ]
    ]

In [11]:
recomendar_hibrido(
    usuario_id=1,
    n=10,
    alpha=0.5
)

,movieId,title,genres,score_hibrido
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,0.981707
1,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,0.822250
2,858,"Godfather, The (1972)",Crime|Drama,0.788118
3,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,0.773600
4,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,0.759344
5,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,0.757549
6,150,Apollo 13 (1995),Adventure|Drama|IMAX,0.755682
7,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery,0.742567
8,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,0.735177
9,4226,Memento (2000),Mystery|Thriller,0.727406


## 7. Análisis del efecto del parámetro alpha

El parámetro `alpha` controla el equilibrio entre personalización y popularidad dentro del modelo híbrido.

Para observar su efecto sobre las recomendaciones se generan diferentes rankings para un mismo usuario modificando progresivamente el peso de ambos componentes.

In [12]:
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]

for alpha in alphas:

    recomendaciones = recomendar_hibrido(
        usuario_id=1,
        n=10,
        alpha=alpha
    )

    print(f"\nAlpha = {alpha}")
    print(
        recomendaciones[
            ["title", "score_hibrido"]
        ].to_string(index=False)
    )


Alpha = 0.0
                                                    title  score_hibrido
                         Shawshank Redemption, The (1994)       0.963415
                        Terminator 2: Judgment Day (1991)       0.679878
                                         Apollo 13 (1995)       0.609756
Lord of the Rings: The Fellowship of the Ring, The (2001)       0.600610
                                    Godfather, The (1972)       0.582317
            Lord of the Rings: The Two Towers, The (2002)       0.570122
    Lord of the Rings: The Return of the King, The (2003)       0.560976
                                           Aladdin (1992)       0.554878
                                  Sixth Sense, The (1999)       0.542683
                                         True Lies (1994)       0.539634

Alpha = 0.25
                                                    title  score_hibrido
                         Shawshank Redemption, The (1994)       0.972561
                        